### Within-Family Diagnostic — BUTTER-E only

Quick diagnostic, not a full model build: train Random Forest (log1p target) on BUTTER-E rows only from `combined_features.csv`, standard random 80/20 split within this family (not pooled with EC-NAS). Purpose: the pooled bake-off's metrics (`03g_model_bakeoff.ipynb`) were low across the board — this checks whether that's a pooling problem or a family-specific one, before adding any new features.

In [ ]:
# IMPORTS & LOAD

import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import random_forest

df = pd.read_csv("../../data/processed/combined/combined_features.csv")
butter = df[df["family"] == "MLP"]

FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]
TARGET = "target"
butter.shape

In [ ]:
# TRAIN & EVALUATE — RF, log1p(target), within-family split only

X = butter[FEATURES]
y = butter[TARGET]
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

model = random_forest.build_model()
model.fit(X_train, y_train_log)
preds_log = model.predict(X_test)
preds_raw = np.expm1(preds_log)

mape_log = mean_absolute_percentage_error(y_test_log, preds_log)
r2_log = r2_score(y_test_log, preds_log)
mape_raw = mean_absolute_percentage_error(y_test, preds_raw)
r2_raw = r2_score(y_test, preds_raw)
tau, tau_p = kendalltau(y_test, preds_raw)  # rank-invariant to monotonic transforms -- same on log or raw scale

print(f"n = {len(butter)}")
print(f"MAPE (log scale):  {mape_log:.4f}")
print(f"R2   (log scale):  {r2_log:.4f}")
print(f"MAPE (raw, expm1): {mape_raw:.4f}")
print(f"R2   (raw, expm1): {r2_raw:.4f}")
print(f"Kendall-Tau:       {tau:.4f}  (p={tau_p:.2e})")

**Result** (n=37,055): MAPE (log) = 0.073, R² (log) = 0.198, MAPE (raw) = 1.290, R² (raw) = 0.110, **Kendall-Tau = 0.274** (p≈3×10⁻²⁷³, highly significant despite the low value).

This is the weak link — see `03b_within_ec_nas.ipynb` and the comparison note there. `params, depth, flops, epochs, batch_size` alone predict BUTTER-E's energy poorly even with no cross-family pooling involved. Plausible missing signal, to investigate before adding features blindly: `is_gpu` (CPU vs. GPU is a large, structural difference in power draw and was never carried into `02a_butter_e_features.ipynb`'s feature set) and `shape`/`dataset` (BUTTER-E's 8 architecture shapes and 12 training datasets — currently collapsed into `params`+`depth` alone, which doesn't capture width/shape variation at a fixed depth, or how energy scales with the training dataset's own size/complexity).

### Re-run with `is_gpu` + one-hot `shape` added

`02a_butter_e_features.ipynb` now adds these two columns (see its updated intro markdown for the core-vs-auxiliary design flag). Loading directly from the rebuilt `data/processed/butter_e/butter_e_features.csv` here rather than `combined_features.csv`, since the combined table hasn't been rebuilt with these columns yet — this is a BUTTER-E-only diagnostic either way.

In [ ]:
# LOAD DIRECT FROM butter_e_features.csv (has is_gpu + shape_* now) & BUILD FEATURE SET

butter_v2 = pd.read_csv("../../data/processed/butter_e/butter_e_features.csv")

SHAPE_COLS = [c for c in butter_v2.columns if c.startswith("shape_")]
FEATURES_V2 = FEATURES + ["is_gpu"] + SHAPE_COLS
print(FEATURES_V2)

X2 = butter_v2[FEATURES_V2]
y2 = butter_v2[TARGET]
y2_log = np.log1p(y2)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
_, _, y2_train_log, y2_test_log = train_test_split(X2, y2_log, test_size=0.2, random_state=42)

In [ ]:
# TRAIN & EVALUATE — same RF, log1p(target), same split logic, expanded feature set

model_v2 = random_forest.build_model()
model_v2.fit(X2_train, y2_train_log)
preds2_log = model_v2.predict(X2_test)
preds2_raw = np.expm1(preds2_log)

mape2_log = mean_absolute_percentage_error(y2_test_log, preds2_log)
r2_2_log = r2_score(y2_test_log, preds2_log)
mape2_raw = mean_absolute_percentage_error(y2_test, preds2_raw)
r2_2_raw = r2_score(y2_test, preds2_raw)
tau2, tau2_p = kendalltau(y2_test, preds2_raw)

print(f"MAPE (log scale):  {mape2_log:.4f}")
print(f"R2   (log scale):  {r2_2_log:.4f}")
print(f"MAPE (raw, expm1): {mape2_raw:.4f}")
print(f"R2   (raw, expm1): {r2_2_raw:.4f}")
print(f"Kendall-Tau:       {tau2:.4f}  (p={tau2_p:.2e})")
print()
print("feature importances:")
print(pd.Series(model_v2.feature_importances_, index=FEATURES_V2).sort_values(ascending=False))

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE (log) | R² (log) | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|---:|---:|
| Baseline (5 core features) | 0.073 | 0.198 | 1.290 | 0.110 | 0.274 |
| + `is_gpu` + one-hot `shape` | 0.070 | 0.248 | 1.128 | **0.075** | **0.319** |

**A real improvement, but not a clean one — same pattern as the log-transform tradeoff in `03g`.** MAPE and Kendall-Tau both improve meaningfully (tau +0.045, a genuine gain in ranking quality; raw MAPE down from 1.29 to 1.13). Log-scale R² improves too (0.198 → 0.248). But **raw-scale R² gets worse** (0.110 → 0.075) — consistent with raw R² being dominated by squared error on the largest energy values, which these new features don't specifically help with.

**Feature importances confirm the hypothesis directly**: `is_gpu` is the single most important feature (30.4%) — more important than `params` or `flops` individually (24.0% each). This was the single biggest gap in the original feature set. The one-hot `shape` columns each contribute modestly (~1% each, ~8% combined) — real but secondary signal. `batch_size` and `epochs` both show 0% importance, as expected — they're constants within BUTTER-E, so the tree has nothing to split on (not a new problem, just confirms those two columns carry no signal for this family specifically).

**Still well short of EC-NAS's 0.861 Kendall-Tau** — this closes some of the gap (0.274 → 0.319) but not most of it. BUTTER-E remains a harder family to predict with the currently-available features; the node-hardware-spec join (`node_sinfo.csv` + GPU/CPU curated specs, already deferred in `02a`) is the next likely candidate, not yet attempted.

**Design-flag resolution:** given `is_gpu` and `shape` don't exist for EC-NAS at all, they should be the **family-specific auxiliary branch** (masked to zero when not applicable), not the shared core set — exactly as the thesis's Section 3.3.2 design already anticipated, and exactly how they're implemented in `02a` above. Keeping the original 9-column schema (`params, depth, flops, epochs, batch_size, target, family, source_dataset, run_id`) as the shared core set for RQ1's cross-family test preserves an important property: a model trained on one family and tested on the other sees the *same* input schema in both directions, with the auxiliary columns simply zeroed out for whichever family they don't apply to. Not yet wired up in `02c_combined_features.ipynb` or the bake-off notebooks — this notebook only establishes that the auxiliary features help BUTTER-E in isolation; masking them into the combined/pooled pipeline is a separate follow-up step.

### Re-run with `memory_fit_ratio` added

`02a_butter_e_features.ipynb` now also adds `memory_fit_ratio` (params × 4 bytes ÷ node RAM in bytes, via a `node_sinfo.csv` join). Same RF, log1p target, same split, on top of the core + `is_gpu` + `shape` feature set from the previous re-run.

In [ ]:
# LOAD (has memory_fit_ratio now) & BUILD FEATURE SET V3

butter_v3 = pd.read_csv("../../data/processed/butter_e/butter_e_features.csv")
FEATURES_V3 = FEATURES_V2 + ["memory_fit_ratio"]
print(FEATURES_V3)

X3 = butter_v3[FEATURES_V3]
y3 = butter_v3[TARGET]
y3_log = np.log1p(y3)

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)
_, _, y3_train_log, y3_test_log = train_test_split(X3, y3_log, test_size=0.2, random_state=42)

model_v3 = random_forest.build_model()
model_v3.fit(X3_train, y3_train_log)
preds3_log = model_v3.predict(X3_test)
preds3_raw = np.expm1(preds3_log)

mape3_log = mean_absolute_percentage_error(y3_test_log, preds3_log)
r2_3_log = r2_score(y3_test_log, preds3_log)
mape3_raw = mean_absolute_percentage_error(y3_test, preds3_raw)
r2_3_raw = r2_score(y3_test, preds3_raw)
tau3, tau3_p = kendalltau(y3_test, preds3_raw)

print(f"MAPE (log scale):  {mape3_log:.4f}")
print(f"R2   (log scale):  {r2_3_log:.4f}")
print(f"MAPE (raw, expm1): {mape3_raw:.4f}")
print(f"R2   (raw, expm1): {r2_3_raw:.4f}")
print(f"Kendall-Tau:       {tau3:.4f}  (p={tau3_p:.2e})")
print()
print("feature importances:")
print(pd.Series(model_v3.feature_importances_, index=FEATURES_V3).sort_values(ascending=False))

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE (log) | R² (log) | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|---:|---:|
| Baseline (5 core features) | 0.073 | 0.198 | 1.290 | 0.110 | 0.274 |
| + `is_gpu` + `shape` | 0.070 | 0.248 | 1.128 | 0.075 | 0.319 |
| + `memory_fit_ratio` | 0.070 | 0.239 | **1.150** | **0.068** | **0.314** |

**`memory_fit_ratio` doesn't help — it's a small net regression on every single metric** (MAPE raw 1.128→1.150, R² raw 0.075→0.068, R² log 0.248→0.239, tau 0.319→0.314). This is despite the model actually using it — feature importance ranks it 4th at 13.9%, ahead of `depth` and taking weight away from `params`/`flops` (which dropped from ~24% each to ~19%/16%).

**Why it likely doesn't help, given the numbers:** `memory_fit_ratio`'s own range is `1.6×10⁻¹⁰` to `7.5×10⁻⁴` (see `02a`) — every single BUTTER-E model's parameter footprint is a tiny fraction of any node's RAM; nothing here ever approaches actual memory pressure. Since `footprint_bytes` is just `params × 4`, the ratio is mostly `params` divided by a denominator (which node a job happened to land on) that's essentially unrelated to energy consumption — `node_sinfo.csv` only reports RAM, not CPU model or clock speed, so there's no compute-relevant signal in the denominator to add. The model still finds *some* structure in it (13.9% importance isn't nothing), but that structure isn't generalizing to the held-out test set — classic redundant-feature-adds-variance behavior in a random forest.

**Recommendation, not yet acted on:** drop `memory_fit_ratio` from the feature set that carries into the pooled bake-off — it doesn't clear the bar the earlier two additions did. `is_gpu` + `shape` (the "+ is_gpu + shape" row above) remains the best BUTTER-E result so far. Not re-running `03g` yet, per instruction — this needs a decision on whether to keep `memory_fit_ratio` in the schema (even if unused) before that happens.

### Re-run with `dataset` added (`memory_fit_ratio` dropped)

Per `03h_diagnostic_feature_audit.ipynb`, `dataset` (one-hot, 12 categories) is promoted into the real feature set — legitimately knowable a priori, unlike `node`/`run_time` which stay exploratory-only. `memory_fit_ratio` is dropped from `02a` entirely (confirmed not helping, previous section). Loading the freshly rebuilt `butter_e_features.csv` (30 columns), same RF/log1p/split as every prior run in this notebook.

In [ ]:
# LOAD (dataset one-hot added, memory_fit_ratio removed) & BUILD FEATURE SET V4

butter_v4 = pd.read_csv("../../data/processed/butter_e/butter_e_features.csv")
DATASET_COLS = [c for c in butter_v4.columns if c.startswith("dataset_")]
FEATURES_V4 = FEATURES + ["is_gpu"] + SHAPE_COLS + DATASET_COLS  # note: no memory_fit_ratio, no node
print(FEATURES_V4)

X4 = butter_v4[FEATURES_V4]
y4 = butter_v4[TARGET]
y4_log = np.log1p(y4)

X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.2, random_state=42)
_, _, y4_train_log, y4_test_log = train_test_split(X4, y4_log, test_size=0.2, random_state=42)

model_v4 = random_forest.build_model()
model_v4.fit(X4_train, y4_train_log)
preds4_log = model_v4.predict(X4_test)
preds4_raw = np.expm1(preds4_log)

mape4_log = mean_absolute_percentage_error(y4_test_log, preds4_log)
r2_4_log = r2_score(y4_test_log, preds4_log)
mape4_raw = mean_absolute_percentage_error(y4_test, preds4_raw)
r2_4_raw = r2_score(y4_test, preds4_raw)
tau4, tau4_p = kendalltau(y4_test, preds4_raw)

print(f"MAPE (log scale):  {mape4_log:.4f}")
print(f"R2   (log scale):  {r2_4_log:.4f}")
print(f"MAPE (raw, expm1): {mape4_raw:.4f}")
print(f"R2   (raw, expm1): {r2_4_raw:.4f}")
print(f"Kendall-Tau:       {tau4:.4f}  (p={tau4_p:.2e})")
print()
print("feature importances:")
print(pd.Series(model_v4.feature_importances_, index=FEATURES_V4).sort_values(ascending=False).head(20))

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE (raw) | R² (raw) | R² (log) | Kendall-Tau |
|---|---:|---:|---:|---:|
| 5 core features | 1.290 | 0.110 | 0.198 | 0.274 |
| + `is_gpu` + `shape` | 1.128 | 0.075 | 0.248 | 0.319 |
| + `memory_fit_ratio` (reverted) | 1.150 | 0.068 | 0.239 | 0.314 |
| **+ `dataset`** | **0.091** | **0.973** | **0.985** | **0.936** |

**Confirms the `03h` finding directly, now with a fully a-priori-safe feature set (no `node`, no `run_time`).** MAPE drops from 113% to 9.1%; R² jumps from 0.075 to 0.973; Kendall-Tau from 0.319 to 0.936. `dataset_sleep` (0.160) and `is_gpu` (0.133) are now the top two features — `is_gpu`'s importance recovered from its near-zero showing in `03h` (0.0015) now that `node_target_enc` isn't competing for the same signal, supporting the earlier read that node identity was partly proxying for GPU/CPU assignment.

**This now exceeds EC-NAS's own within-family result** (`03b_within_ec_nas.ipynb`: R²=0.958, tau=0.861) — BUTTER-E's tau of 0.936 is higher, on a dataset roughly 13x larger (37,055 vs 2,805 rows). Both families are now well-predicted from legitimately a-priori-knowable features; the `node`/cluster-contention ceiling identified in `03h` (tau contribution ≈0.159) remains a real, separate limitation, but it's no longer the practical bottleneck it looked like before `dataset` was found.

Not yet re-run: `03g_model_bakeoff.ipynb` (pooled bake-off) — per instruction, holding off until this `03a` improvement was confirmed, which it now is.

### Re-run with dataset *properties* instead of dataset *identity*

`02a_butter_e_features.ipynb` now replaces the 12 one-hot `dataset_*` columns with 5 numeric properties joined from `pmlb.csv` (`n_observations`, `n_features`, `n_classes`, `task_encoded`, `imbalance`) — for generalization to datasets never seen during training, which one-hot identity can't provide. Same RF/log1p/split.

In [ ]:
# LOAD (dataset properties instead of one-hot identity) & BUILD FEATURE SET V5

butter_v5 = pd.read_csv("../../data/processed/butter_e/butter_e_features.csv")
PROPERTY_COLS = ["n_observations", "n_features", "n_classes", "task_encoded", "imbalance"]
FEATURES_V5 = FEATURES + ["is_gpu"] + SHAPE_COLS + PROPERTY_COLS
print(FEATURES_V5)

X5 = butter_v5[FEATURES_V5]
y5 = butter_v5[TARGET]
y5_log = np.log1p(y5)

X5_train, X5_test, y5_train, y5_test = train_test_split(X5, y5, test_size=0.2, random_state=42)
_, _, y5_train_log, y5_test_log = train_test_split(X5, y5_log, test_size=0.2, random_state=42)

model_v5 = random_forest.build_model()
model_v5.fit(X5_train, y5_train_log)
preds5_log = model_v5.predict(X5_test)
preds5_raw = np.expm1(preds5_log)

mape5_log = mean_absolute_percentage_error(y5_test_log, preds5_log)
r2_5_log = r2_score(y5_test_log, preds5_log)
mape5_raw = mean_absolute_percentage_error(y5_test, preds5_raw)
r2_5_raw = r2_score(y5_test, preds5_raw)
tau5, tau5_p = kendalltau(y5_test, preds5_raw)

print(f"MAPE (log scale):  {mape5_log:.4f}")
print(f"R2   (log scale):  {r2_5_log:.4f}")
print(f"MAPE (raw, expm1): {mape5_raw:.4f}")
print(f"R2   (raw, expm1): {r2_5_raw:.4f}")
print(f"Kendall-Tau:       {tau5:.4f}  (p={tau5_p:.2e})")
print()
print("feature importances:")
print(pd.Series(model_v5.feature_importances_, index=FEATURES_V5).sort_values(ascending=False))

**Result** (executed once already; re-run in VS Code to attach outputs):

| feature set | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|
| + `is_gpu` + `shape` only | 1.128 | 0.075 | 0.319 |
| + one-hot `dataset` (12 cols) | 0.091 | 0.973 | 0.936 |
| **+ dataset properties (5 cols)** | **0.091** | **0.973** | **0.936** |

**Matches the one-hot result almost exactly (0.0909 vs 0.0910 MAPE, 0.9728 vs 0.9726 R², 0.9359 vs 0.9357 tau) with fewer than half the columns (5 vs 12) — and unlike one-hot, these 5 generalize to a dataset BUTTER-E never trained on.** A held-out 13th PMLB dataset, or any dataset outside PMLB entirely, still gets meaningful, non-zero feature values here; a one-hot identity column would just be all-zero for it (no signal at all, not even a reasonable default).

**Feature importance is dominated almost entirely by `n_observations` alone (0.595)** — larger by 4x than the next feature (`is_gpu`, 0.132). `n_features` (0.011), `imbalance` (0.004), and `n_classes` (0.003) each contribute comparatively little; `task_encoded` (classification vs. regression) is nearly zero (0.0006). This is a clean, interpretable finding: the mechanism behind `03h`'s `dataset` result is overwhelmingly **dataset size** — more training instances mechanically means more per-epoch compute time, hence more energy — not task type or class structure. `n_observations` alone is very likely close to sufficient; `n_features`/`n_classes`/`imbalance`/`task_encoded` are along for completeness rather than doing real work here.